# Preparazione del dataset



In [34]:
!pip install lxml

## 1. Variazioni testuali di due testimoni manoscritti codificati in TEI-XML



In [35]:
from google.colab import files
uploaded = files.upload()

Saving merged_manuscripts.xml to merged_manuscripts (3).xml


In [36]:
from lxml import etree
import pandas as pd
import re

tree = etree.parse("merged_manuscripts.xml")
root = tree.getroot()

NS = {"tei": "http://www.tei-c.org/ns/1.0"}

In [37]:
divs = root.findall(".//tei:div", namespaces=NS)

## 2. Estrazione e normalizzazione del testo

Il testo viene estratto dal file XML-TEI preservando il contenuto delle espansioni abbreviative ed escludendo i commenti.
La normalizzazione viene applicata solo alla colonna `text_normalized`, usata per la valutazione computazionale della similarità. La colonna `text_expanded` conserva invece il testo estratto in forma leggibile.

Durante l’estrazione, alcune parole risultano separate perché nel manoscritto sono divise da fine riga e per il confronto computazionale vengono ricomposti selettivamente nella versione normalizzata, senza modificare il testo espanso.

In [38]:
def extract_tei_text(element):
    """
    Extract readable text from a TEI element, preserving text inside
    abbreviation expansions and skipping XML comments.
    """
    parts = []

    if element.text:
        parts.append(element.text)

    for child in element:
        if isinstance(child, etree._Comment):
            if child.tail:
                parts.append(child.tail)
            continue

        parts.append(extract_tei_text(child))

        if child.tail:
            parts.append(child.tail)

    text = "".join(parts)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_text(text):
    """
    Apply light normalization for computational comparison.

    The original expanded text is not modified. This normalized version is used
    only for similarity evaluation.
    """
    text = text.lower()

    # Recompose selected word splits caused by line breaks or word division
    # in the manuscript/transcription. These replacements affect only the
    # normalized text used for computation, not the expanded text.
    line_break_recompositions = {
        "con cilium": "concilium",
        "difficul tates": "difficultates",
        "obe diunt": "obediunt",
        "im perator": "imperator"
    }

    for split_form, recomposed_form in line_break_recompositions.items():
        text = text.replace(split_form, recomposed_form)

    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

Verifico che effettivamente la funzione `normalize_text` stia ricomponendo correttamente le parole divise.




In [39]:
normalization_examples = {
    "con cilium": normalize_text("con cilium"),
    "difficul tates": normalize_text("difficul tates"),
    "obe diunt": normalize_text("obe diunt"),
    "Im perator": normalize_text("Im perator")
}

normalization_examples

{'con cilium': 'concilium',
 'difficul tates': 'difficultates',
 'obe diunt': 'obediunt',
 'Im perator': 'imperator'}

## 3. Creazione del dataset

In [40]:
section_titles = {
    "wit_D54": "Nota quod Sacro concilio non est detrahendum patet ex multis",
    "wit_4941": "De sacro Concilio non est detrahendum patet ex multis"
}

manuscript_labels = {
    "wit_D54": "Prague D.54",
    "wit_4941": "Wien 4941"
}

rows = []

for d in divs:
    witness = d.get("{http://www.w3.org/XML/1998/namespace}id")
    manuscript = manuscript_labels.get(witness, witness)
    section_title = section_titles.get(witness, "")

    first_p = d.find(".//tei:p", namespaces=NS)
    segments = first_p.findall(".//tei:seg[@type='argument']", namespaces=NS)

    for seg in segments:
        text_expanded = extract_tei_text(seg)
        text_normalized = normalize_text(text_expanded)

        rows.append({
            "witness": witness,
            "manuscript": manuscript,
            "section_title": section_title,
            "argument_n": seg.get("n"),
            "text_expanded": text_expanded,
            "text_normalized": text_normalized
        })

df_segments = pd.DataFrame(rows)
df_segments

,witness,manuscript,section_title,argument_n,text_expanded,text_normalized
0,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,1,primo apostoli illud in instituunt et celebravunt,primo apostoli illud in instituunt et celebravunt
1,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,2,Secundo quia per sacrorum concilium quattuor s...,secundo quia per sacrorum concilium quattuor s...
2,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,3,Tercio Sacrum con cilium solet terminare omnes...,tercio sacrum concilium solet terminare omnes ...
3,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,4,Quarto quia papa et Imperator confirmarunt con...,quarto quia papa et imperator confirmarunt con...
4,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,5,Quinto quia omnes Reges christianitatis et omn...,quinto quia omnes reges christianitatis et omn...
5,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,6,Sexto quia nullus tam magnus est qui presumat ...,sexto quia nullus tam magnus est qui presumat ...
6,wit_4941,Wien 4941,De sacro Concilio non est detrahendum patet ex...,1,Primo quia Apostoli ide instituerunt et celebr...,primo quia apostoli ide instituerunt et celebr...
7,wit_4941,Wien 4941,De sacro Concilio non est detrahendum patet ex...,2,secundo quia per sacrum Concilium quator sanct...,secundo quia per sacrum concilium quator sanct...
8,wit_4941,Wien 4941,De sacro Concilio non est detrahendum patet ex...,3,Tercio sacrum Concilium solet terminare omnes ...,tercio sacrum concilium solet terminare omnes ...
9,wit_4941,Wien 4941,De sacro Concilio non est detrahendum patet ex...,4,Quarto quia papa et Imperator confirmaventur C...,quarto quia papa et imperator confirmaventur c...


## 4. Preview ed esportazione

In [41]:
df_segments[["manuscript", "argument_n", "text_expanded", "text_normalized"]]

,manuscript,argument_n,text_expanded,text_normalized
0,Prague D.54,1,primo apostoli illud in instituunt et celebravunt,primo apostoli illud in instituunt et celebravunt
1,Prague D.54,2,Secundo quia per sacrorum concilium quattuor s...,secundo quia per sacrorum concilium quattuor s...
2,Prague D.54,3,Tercio Sacrum con cilium solet terminare omnes...,tercio sacrum concilium solet terminare omnes ...
3,Prague D.54,4,Quarto quia papa et Imperator confirmarunt con...,quarto quia papa et imperator confirmarunt con...
4,Prague D.54,5,Quinto quia omnes Reges christianitatis et omn...,quinto quia omnes reges christianitatis et omn...
5,Prague D.54,6,Sexto quia nullus tam magnus est qui presumat ...,sexto quia nullus tam magnus est qui presumat ...
6,Wien 4941,1,Primo quia Apostoli ide instituerunt et celebr...,primo quia apostoli ide instituerunt et celebr...
7,Wien 4941,2,secundo quia per sacrum Concilium quator sanct...,secundo quia per sacrum concilium quator sanct...
8,Wien 4941,3,Tercio sacrum Concilium solet terminare omnes ...,tercio sacrum concilium solet terminare omnes ...
9,Wien 4941,4,Quarto quia papa et Imperator confirmaventur C...,quarto quia papa et imperator confirmaventur c...


In [42]:
df_segments.to_csv("segments_dataset.csv", index=False, encoding="utf-8")
files.download("segments_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Similarity Evaluation

Questa sezione confronta il testo delle sei argomentazioni nei due testimoni (Prague D.54 e Wien 4941), quindi a livello degli elementi TEI `<seg type="argument">`.

Vengono utilizzate due misure di similarità: **Jaccard similarity**, che valuta la sovrapposizione lessicale indipendentemente dall’ordine delle parole, e la **Longest Common Subsequence similarity**, che valuta quanto del materiale testuale compare nello stesso ordine nei due testimoni.

Le misure di similarità sono calcolate sulla colonna `text_normalized`, non sulla colonna `text_expanded`. Questo permette di ridurre l’effetto computazionale di fenomeni grafici come la divisione di parola a fine riga, mantenendo comunque separato il testo estratto dalla sua forma normalizzata.

In [43]:
def jaccard_similarity(text_a, text_b):
    """
    Compute Jaccard similarity between two texts.
    The texts are represented as sets of tokens.
    """
    A = set(text_a.split())
    B = set(text_b.split())

    intersection = A & B
    union = A | B

    if len(union) == 0:
        return 1.0

    return len(intersection) / len(union)


def longest_common_subsequence(seq_a, seq_b):
    """
    Compute the Longest Common Subsequence between two token sequences.
    """
    m = len(seq_a)
    n = len(seq_b)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq_a[i - 1] == seq_b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    # Reconstruct LCS
    i, j = m, n
    lcs = []

    while i > 0 and j > 0:
        if seq_a[i - 1] == seq_b[j - 1]:
            lcs.append(seq_a[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1

    lcs.reverse()
    return lcs


def lcs_similarity(text_a, text_b):
    """
    Compute normalized LCS similarity between two texts.
    The score is the length of the LCS divided by the maximum sequence length.
    """
    seq_a = text_a.split()
    seq_b = text_b.split()

    if len(seq_a) == 0 and len(seq_b) == 0:
        return 1.0

    lcs = longest_common_subsequence(seq_a, seq_b)
    return len(lcs) / max(len(seq_a), len(seq_b))

In [44]:
similarity_rows = []

for argument_n in sorted(df_segments["argument_n"].unique(), key=int):

    d54_text = df_segments[
        (df_segments["witness"] == "wit_D54") &
        (df_segments["argument_n"] == argument_n)
    ]["text_normalized"].iloc[0]

    wien_text = df_segments[
        (df_segments["witness"] == "wit_4941") &
        (df_segments["argument_n"] == argument_n)
    ]["text_normalized"].iloc[0]

    jaccard = jaccard_similarity(d54_text, wien_text)
    lcs = lcs_similarity(d54_text, wien_text)

    similarity_rows.append({
        "argument_n": argument_n,
        "jaccard_similarity": round(jaccard, 3),
        "lcs_similarity": round(lcs, 3),
        "D54_length_tokens": len(d54_text.split()),
        "4941_length_tokens": len(wien_text.split())
    })

df_similarity = pd.DataFrame(similarity_rows)
df_similarity

,argument_n,jaccard_similarity,lcs_similarity,D54_length_tokens,4941_length_tokens
0,1,0.273,0.429,7,7
1,2,0.760,0.857,28,28
2,3,0.875,0.609,23,14
3,4,0.882,0.944,18,18
4,5,0.632,0.783,20,23
5,6,0.913,0.909,22,22


In [45]:
def interpret_similarity(jaccard, lcs):
    """
    Provide a simple qualitative interpretation of similarity scores.
    """
    if jaccard >= 0.75 and lcs >= 0.75:
        return "high lexical and sequential similarity"
    elif jaccard >= 0.75 and lcs < 0.75:
        return "high lexical overlap, lower sequential similarity"
    elif jaccard < 0.75 and lcs >= 0.75:
        return "lower lexical overlap, but stable sequence"
    else:
        return "lower similarity; possible omission, addition, or reformulation"


df_similarity["interpretation"] = df_similarity.apply(
    lambda row: interpret_similarity(
        row["jaccard_similarity"],
        row["lcs_similarity"]
    ),
    axis=1
)

df_similarity

,argument_n,jaccard_similarity,lcs_similarity,D54_length_tokens,4941_length_tokens,interpretation
0,1,0.273,0.429,7,7,"lower similarity; possible omission, addition,..."
1,2,0.760,0.857,28,28,high lexical and sequential similarity
2,3,0.875,0.609,23,14,"high lexical overlap, lower sequential similarity"
3,4,0.882,0.944,18,18,high lexical and sequential similarity
4,5,0.632,0.783,20,23,"lower lexical overlap, but stable sequence"
5,6,0.913,0.909,22,22,high lexical and sequential similarity


In [46]:
df_similarity.to_csv("similarity_by_segment.csv", index=False, encoding="utf-8")
files.download("similarity_by_segment.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Interpretazione dei casi selezionati

I punteggi che seguono servono a selezionare alcuni casi significativi, più o meno stabili, da osservare manualmente per indagare la variazione tra i testimoni.


In [47]:
comparison_rows = []

for argument_n in sorted(df_segments["argument_n"].unique(), key=int):

    d54_row = df_segments[
        (df_segments["witness"] == "wit_D54") &
        (df_segments["argument_n"] == argument_n)
    ].iloc[0]

    wien_row = df_segments[
        (df_segments["witness"] == "wit_4941") &
        (df_segments["argument_n"] == argument_n)
    ].iloc[0]

    similarity_row = df_similarity[
        df_similarity["argument_n"] == argument_n
    ].iloc[0]

    comparison_rows.append({
        "argument_n": argument_n,
        "D54_text": d54_row["text_normalized"],
        "4941_text": wien_row["text_normalized"],
        "jaccard_similarity": similarity_row["jaccard_similarity"],
        "lcs_similarity": similarity_row["lcs_similarity"],
        "interpretation": similarity_row["interpretation"]
    })

df_comparison = pd.DataFrame(comparison_rows)
df_comparison

,argument_n,D54_text,4941_text,jaccard_similarity,lcs_similarity,interpretation
0,1,primo apostoli illud in instituunt et celebravunt,primo quia apostoli ide instituerunt et celebr...,0.273,0.429,"lower similarity; possible omission, addition,..."
1,2,secundo quia per sacrorum concilium quattuor s...,secundo quia per sacrum concilium quator sanct...,0.760,0.857,high lexical and sequential similarity
2,3,tercio sacrum concilium solet terminare omnes ...,tercio sacrum concilium solet terminare omnes ...,0.875,0.609,"high lexical overlap, lower sequential similarity"
3,4,quarto quia papa et imperator confirmarunt con...,quarto quia papa et imperator confirmaventur c...,0.882,0.944,high lexical and sequential similarity
4,5,quinto quia omnes reges christianitatis et omn...,quinto quia omnes reges christianitatis et omn...,0.632,0.783,"lower lexical overlap, but stable sequence"
5,6,sexto quia nullus tam magnus est qui presumat ...,sexto quia nullus est tam magnus qui presumat ...,0.913,0.909,high lexical and sequential similarity


## 6.1 Analisi dei segmenti più stabili (6, 4 e 2)

Analizzando la tabella vediamo che i segmenti i con valori più alti, sono 6, 4 e 2.

L’argomento 6 presenta il valore più alto della Jaccard similarity (0.913) e un valore molto alto di LCS similarity (0.909). Questo indica che i due testimoni condividono quasi lo stesso materiale lessicale (notiamo la variante grafica *concilium/consilium*) e mantengono una sequenza testuale molto simile (*est* prima e dopo *tam magnus*), con pochissime differenze che non alterano la struttura complessiva del segmento.

L’argomento 4 presenta invece il valore più alto di LCS similarity (0.944), insieme a una Jaccard similarity molto alta (0.882), a indicare che i due testimoni conservano lo stesso ordine testuale, infatti cambia solo il tempo verbale *confirmarunt/confirmaventur*.

Anche l’argomento 2 mostra valori abbastanza alti in entrambe le misure: Jaccard = 0.760 e LCS = 0.857. In questo caso, la sequenza rimane intatta, cambiano le varianti grafiche o morfologiche.

## 6.2 Analisi dei segmenti più variabili (1, 3 e 5)

Invece, sono gli argomenti 1, 3 e 5 a mostrare i risultati più problematici.

Quello con i valori più bassi è l'argomento 1: Jaccard similarity = 0.273 e LCS similarity = 0.429. Tuttavia, questo segmento è molto breve, con 7 token in entrambi i testimoni. Di conseguenza, le misure sono particolarmente sensibili anche a piccole variazioni: da differenza tra forme come *instituunt/instituerunt* e *celebravunt/celebrarunt* pesano sul punteggio abbassandolo.

L’argomento 3 è particolarmente interessante perché presenta una Jaccard similarity alta (0.875), ma una LCS similarity più bassa (0.609), e questo significa che i due testimoni condividono molto materiale lessicale, ma non lo conservano nella stessa estensione sequenziale. Inoltre, la differenza di lunghezza è significativa: 4941 contiene 14 token, mentre D54 23 poichè include anche la parte *et dampnare omnes hereses que surgunt in ecclesia dei*. Il risultato suggerisce quindi una omissione o compressione in 4941.

L’argomento 5 mostra un caso diverso: la Jaccard similarity è più bassa (0.632), mentre la LCS similarity è relativamente alta (0.783). Questo suggerisce che la sequenza resta riconoscibile, ma con differenze lessicali e con materiale aggiuntivo in 4941: il troncamento di *episcopi* nel testimone 1 e l'aggiunta della formula finale *quod nos obediamus* nel testimone 2.

In [48]:
df_comparison.to_csv("comparison_by_segment.csv", index=False, encoding="utf-8")
files.download("comparison_by_segment.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Edit distance e allineamento esplorativo

L'obiettivo di questa sezione è applicare la Levenshtein distance per misurare il numero minimo di operazioni (es. sostituzioni, cancellazioni) necessarie per trasformare il segmento di D54 nel segmento corrispondente di 4941.

Successivamente, si osserverà un singolo caso viene attraverso un piccolo allineamento esplorativo.

In [49]:
def levenshtein_distance_tokens(seq_a, seq_b):
    """
    Compute Levenshtein distance between two token sequences.

    The allowed operations are insertion, deletion and substitution.
    """
    m = len(seq_a)
    n = len(seq_b)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i

    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if seq_a[i - 1] == seq_b[j - 1] else 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,       # deletion
                dp[i][j - 1] + 1,       # insertion
                dp[i - 1][j - 1] + cost # substitution
            )

    return dp[m][n]


def normalized_levenshtein_similarity(text_a, text_b):
    """
    Convert token-level Levenshtein distance into a similarity score between 0 and 1.
    """
    seq_a = text_a.split()
    seq_b = text_b.split()

    max_len = max(len(seq_a), len(seq_b))

    if max_len == 0:
        return 0, 1.0

    distance = levenshtein_distance_tokens(seq_a, seq_b)
    similarity = 1 - (distance / max_len)

    return distance, similarity

In [50]:
levenshtein_rows = []

for argument_n in sorted(df_segments["argument_n"].unique(), key=int):

    d54_text = df_segments[
        (df_segments["witness"] == "wit_D54") &
        (df_segments["argument_n"] == argument_n)
    ]["text_normalized"].iloc[0]

    wien_text = df_segments[
        (df_segments["witness"] == "wit_4941") &
        (df_segments["argument_n"] == argument_n)
    ]["text_normalized"].iloc[0]

    distance, similarity = normalized_levenshtein_similarity(d54_text, wien_text)

    levenshtein_rows.append({
        "argument_n": argument_n,
        "levenshtein_distance": distance,
        "levenshtein_similarity": round(similarity, 3)
    })

df_levenshtein = pd.DataFrame(levenshtein_rows)
df_levenshtein

,argument_n,levenshtein_distance,levenshtein_similarity
0,1,5,0.286
1,2,4,0.857
2,3,9,0.609
3,4,1,0.944
4,5,5,0.783
5,6,3,0.864


In [51]:
df_similarity_extended = df_similarity.merge(
    df_levenshtein,
    on="argument_n",
    how="left"
)

df_similarity_extended

,argument_n,jaccard_similarity,lcs_similarity,D54_length_tokens,4941_length_tokens,interpretation,levenshtein_distance,levenshtein_similarity
0,1,0.273,0.429,7,7,"lower similarity; possible omission, addition,...",5,0.286
1,2,0.760,0.857,28,28,high lexical and sequential similarity,4,0.857
2,3,0.875,0.609,23,14,"high lexical overlap, lower sequential similarity",9,0.609
3,4,0.882,0.944,18,18,high lexical and sequential similarity,1,0.944
4,5,0.632,0.783,20,23,"lower lexical overlap, but stable sequence",5,0.783
5,6,0.913,0.909,22,22,high lexical and sequential similarity,3,0.864


### 7.1 Interpretazione dell’edit distance

I risultati ci lasciano intendere che Levenshtein distance ha confermato la similarity evaluation, dove infatti gli argomenti 4, 2 e 6 dimostrano la distanza più bassa tra i testimoni. Questa analisi ci permette indubbiamente di notare poi l'alto costo di trasformazione di alcuni segmenti, come per esempio nel caso dell'arg 3,  dove infatti la Levenshtein distance è 9.

In [52]:
df_similarity_extended.to_csv("similarity_extended_by_segment.csv", index=False, encoding="utf-8")
files.download("similarity_extended_by_segment.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Esempio di allineamento: argument 3

Dopo aver calcolato la Levenshtein distance su tutti i segmenti, osserviamo più da vicino un caso specifico: l’argument 3.

Questo segmento è stato scelto perché presenta una combinazione interessante: la Jaccard similarity è alta, ma la LCS similarity e la Levenshtein similarity sono più basse; questo suggerisce che i due testimoni condividono molto materiale lessicale, ma differiscono nella continuità sequenziale e nella lunghezza del segmento.

L’allineamento in questione vuole essere un esempio esplorativo per rendere visibile il rapporto tra punteggi quantitativi e variazione testuale locale.

In [53]:
argument_to_align = "3"

d54_argument = df_segments[
    (df_segments["witness"] == "wit_D54") &
    (df_segments["argument_n"] == argument_to_align)
]["text_normalized"].iloc[0]

wien_argument = df_segments[
    (df_segments["witness"] == "wit_4941") &
    (df_segments["argument_n"] == argument_to_align)
]["text_normalized"].iloc[0]

print("D54 argument 3:")
print(d54_argument)

print("\n4941 argument 3:")
print(wien_argument)

D54 argument 3:
tercio sacrum concilium solet terminare omnes causas et difficultates que in ecclesia dei surgunt et dampnare omnes hereses que surgunt in ecclesia dei

4941 argument 3:
tercio sacrum concilium solet terminare omnes causas et difficultates que surgunt in ecclesia dei


In [54]:
def needleman_wunsch_tokens(seq_a, seq_b, match_score=1, mismatch_score=-1, gap_score=-1):
    """
    Perform a simple Needleman-Wunsch global alignment on two token sequences.
    """

    m = len(seq_a)
    n = len(seq_b)

    # Initialize scoring matrix
    score = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        score[i][0] = score[i - 1][0] + gap_score

    for j in range(1, n + 1):
        score[0][j] = score[0][j - 1] + gap_score

    # Fill scoring matrix
    for i in range(1, m + 1):
        for j in range(1, n + 1):

            if seq_a[i - 1] == seq_b[j - 1]:
                diagonal = score[i - 1][j - 1] + match_score
            else:
                diagonal = score[i - 1][j - 1] + mismatch_score

            up = score[i - 1][j] + gap_score
            left = score[i][j - 1] + gap_score

            score[i][j] = max(diagonal, up, left)

    # Traceback
    aligned_a = []
    aligned_b = []

    i = m
    j = n

    while i > 0 or j > 0:
        current = score[i][j]

        if i > 0 and j > 0:
            if seq_a[i - 1] == seq_b[j - 1]:
                diagonal_score = match_score
            else:
                diagonal_score = mismatch_score

            if current == score[i - 1][j - 1] + diagonal_score:
                aligned_a.append(seq_a[i - 1])
                aligned_b.append(seq_b[j - 1])
                i -= 1
                j -= 1
                continue

        if i > 0 and current == score[i - 1][j] + gap_score:
            aligned_a.append(seq_a[i - 1])
            aligned_b.append("-")
            i -= 1
        else:
            aligned_a.append("-")
            aligned_b.append(seq_b[j - 1])
            j -= 1

    aligned_a.reverse()
    aligned_b.reverse()

    return aligned_a, aligned_b, score[m][n]

In [55]:
seq_d54 = d54_argument.split()
seq_4941 = wien_argument.split()

aligned_d54, aligned_4941, alignment_score = needleman_wunsch_tokens(
    seq_d54,
    seq_4941
)

alignment_rows = []

for position, (token_d54, token_4941) in enumerate(zip(aligned_d54, aligned_4941), start=1):

    if token_d54 == token_4941:
        operation = "match"
    elif token_d54 == "-":
        operation = "insertion_in_4941"
    elif token_4941 == "-":
        operation = "omission_in_4941"
    else:
        operation = "substitution"

    alignment_rows.append({
        "argument_n": argument_to_align,
        "position": position,
        "D54_token": token_d54,
        "4941_token": token_4941,
        "operation": operation
    })

df_argument3_alignment = pd.DataFrame(alignment_rows)

print("Alignment score:", alignment_score)

df_argument3_alignment

Alignment score: 5


,argument_n,position,D54_token,4941_token,operation
0,3,1,tercio,tercio,match
1,3,2,sacrum,sacrum,match
2,3,3,concilium,concilium,match
3,3,4,solet,solet,match
4,3,5,terminare,terminare,match
5,3,6,omnes,omnes,match
6,3,7,causas,causas,match
7,3,8,et,et,match
8,3,9,difficultates,difficultates,match
9,3,10,que,-,omission_in_4941


In [56]:
df_argument3_differences = df_argument3_alignment[
    df_argument3_alignment["operation"] != "match"
]

df_argument3_differences

,argument_n,position,D54_token,4941_token,operation
9,3,10,que,-,omission_in_4941
10,3,11,in,-,omission_in_4941
11,3,12,ecclesia,-,omission_in_4941
12,3,13,dei,-,omission_in_4941
13,3,14,surgunt,-,omission_in_4941
14,3,15,et,-,omission_in_4941
15,3,16,dampnare,-,omission_in_4941
16,3,17,omnes,-,omission_in_4941
17,3,18,hereses,-,omission_in_4941


In [57]:
df_argument3_alignment.to_csv("argument3_alignment.csv", index=False, encoding="utf-8")
files.download("argument3_alignment.csv")

df_argument3_differences.to_csv("argument3_alignment_differences.csv", index=False, encoding="utf-8")
files.download("argument3_alignment_differences.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 8.1 Interpretazione dell’allineamento

L’allineamento dell’argument 3 va a confermare quella discrepanza tra i due testimoni, per cui condividono l’inizio del segmento *tercio sacrum concilium solet terminare omnes causas et difficultates* ma il D54 conserva una formulazione più estesa rispetto a 4941. L’argument 3 quindi presenta un’elevata sovrapposizione lessicale, ma una minore continuità sequenziale.

In particolare, nel primo testimone emerge una porzione relativa alla condanna delle eresie *et dampnare omnes hereses que surgunt in ecclesia dei*, che forse non è stata considerata opportuna nel secondo testimone, o al contrario è stata aggiunta nel primo come specifica correttiva.

Va osservato che, in presenza di sequenze ripetute come *que surgunt in ecclesia dei*, l’allineamento automatico può scegliere una corrispondenza tra occorrenze diverse della stessa formula.

## 9. Classificazione linguistica delle varianti

Dopo la similarity evaluation e l’esempio di allineamento, questa sezione organizza alcune varianti osservate in una piccola tabella con le principali varianti osservate nei sei segmenti argomentativi confrontati. La classificazione rende esplicito il passaggio dai risultati computazionali all’interpretazione linguistica e filologica.

La classificazione distingue tra diversi livelli di variazione:

- variazione grafica o ortografica;
- variazione morfologica;
- variazione lessicale;
- variazione d’ordine;
- aggiunta;
- omissione o compressione.


In [58]:
variant_rows = [
    # Argument 1
    {
        "argument_n": "1",
        "D54_reading": "illud in",
        "4941_reading": "quia",
        "variation_type": "lexical variation",
        "linguistic_level": "lexical/textual",
        "effect_on_similarity": "lowers Jaccard and Levenshtein similarity",
        "note": "Different wording in a very short segment."
    },
    {
        "argument_n": "1",
        "D54_reading": "instituunt",
        "4941_reading": "instituerunt",
        "variation_type": "morphological variation",
        "linguistic_level": "morphological",
        "effect_on_similarity": "lowers all token-based similarity measures",
        "note": "Different verbal form."
    },
    {
        "argument_n": "1",
        "D54_reading": "celebravunt",
        "4941_reading": "celebrarunt",
        "variation_type": "morphological/graphic variation",
        "linguistic_level": "morphological/orthographic",
        "effect_on_similarity": "lowers all token-based similarity measures",
        "note": "Different verbal surface form."
    },

    # Argument 2
    {
        "argument_n": "2",
        "D54_reading": "sacrorum",
        "4941_reading": "sacrum",
        "variation_type": "morphological variation",
        "linguistic_level": "morphological",
        "effect_on_similarity": "minor effect on token-based similarity",
        "note": "Different inflected form."
    },
    {
        "argument_n": "2",
        "D54_reading": "quattuor",
        "4941_reading": "quator",
        "variation_type": "graphic/orthographic variation",
        "linguistic_level": "orthographic",
        "effect_on_similarity": "minor effect on Jaccard similarity",
        "note": "Different spelling of the same numeral."
    },
    {
        "argument_n": "2",
        "D54_reading": "quibi",
        "4941_reading": "quibus",
        "variation_type": "morphological/graphic variation",
        "linguistic_level": "morphological/orthographic",
        "effect_on_similarity": "minor effect on token-based similarity",
        "note": "Different surface form of the relative pronoun."
    },
    {
        "argument_n": "2",
        "D54_reading": "reiciun",
        "4941_reading": "reiciunt",
        "variation_type": "graphic/morphological variation",
        "linguistic_level": "orthographic/morphological",
        "effect_on_similarity": "minor effect on token-based similarity",
        "note": "Final consonant is absent in D54."
    },

    # Argument 3
    {
        "argument_n": "3",
        "D54_reading": "que in ecclesia dei surgunt et dampnare omnes hereses",
        "4941_reading": "-",
        "variation_type": "omission/compression",
        "linguistic_level": "textual",
        "effect_on_similarity": "lowers LCS and Levenshtein similarity",
        "note": "Material present in D54 but absent in the shorter version of 4941."
    },

    # Argument 4
    {
        "argument_n": "4",
        "D54_reading": "confirmarunt",
        "4941_reading": "confirmaventur",
        "variation_type": "morphological/lexical variation",
        "linguistic_level": "morphological/lexical",
        "effect_on_similarity": "minor effect; segment remains highly stable",
        "note": "Different verbal form."
    },

    # Argument 5
    {
        "argument_n": "5",
        "D54_reading": "principes",
        "4941_reading": "pricipes",
        "variation_type": "graphic/orthographic variation",
        "linguistic_level": "orthographic",
        "effect_on_similarity": "lowers Jaccard similarity",
        "note": "Different spelling of the same lexical item."
    },
    {
        "argument_n": "5",
        "D54_reading": "scopi",
        "4941_reading": "episcopi",
        "variation_type": "lexical or abbreviation-related variation",
        "linguistic_level": "lexical/orthographic",
        "effect_on_similarity": "lowers Jaccard similarity",
        "note": "The two forms are related but do not match as identical tokens."
    },
    {
        "argument_n": "5",
        "D54_reading": "-",
        "4941_reading": "quod nos obediamus",
        "variation_type": "addition",
        "linguistic_level": "textual",
        "effect_on_similarity": "lowers Jaccard similarity and increases length of 4941",
        "note": "Additional final formula in 4941."
    },

    # Argument 6
    {
        "argument_n": "6",
        "D54_reading": "nullus tam magnus est",
        "4941_reading": "nullus est tam magnus",
        "variation_type": "word order variation",
        "linguistic_level": "syntactic/sequential",
        "effect_on_similarity": "slightly lowers sequence-based similarity",
        "note": "The lexical material is the same, but the order changes locally."
    },
    {
        "argument_n": "6",
        "D54_reading": "concilio",
        "4941_reading": "consilio",
        "variation_type": "graphic/orthographic or lexical variation",
        "linguistic_level": "orthographic/lexical",
        "effect_on_similarity": "minor effect; segment remains highly similar",
        "note": "The variation affects the surface token."
    }
]

df_variant_classification = pd.DataFrame(variant_rows)
df_variant_classification

,argument_n,D54_reading,4941_reading,variation_type,linguistic_level,effect_on_similarity,note
0,1,illud in,quia,lexical variation,lexical/textual,lowers Jaccard and Levenshtein similarity,Different wording in a very short segment.
1,1,instituunt,instituerunt,morphological variation,morphological,lowers all token-based similarity measures,Different verbal form.
2,1,celebravunt,celebrarunt,morphological/graphic variation,morphological/orthographic,lowers all token-based similarity measures,Different verbal surface form.
3,2,sacrorum,sacrum,morphological variation,morphological,minor effect on token-based similarity,Different inflected form.
4,2,quattuor,quator,graphic/orthographic variation,orthographic,minor effect on Jaccard similarity,Different spelling of the same numeral.
5,2,quibi,quibus,morphological/graphic variation,morphological/orthographic,minor effect on token-based similarity,Different surface form of the relative pronoun.
6,2,reiciun,reiciunt,graphic/morphological variation,orthographic/morphological,minor effect on token-based similarity,Final consonant is absent in D54.
7,3,que in ecclesia dei surgunt et dampnare omnes ...,-,omission/compression,textual,lowers LCS and Levenshtein similarity,Material present in D54 but absent in the shor...
8,4,confirmarunt,confirmaventur,morphological/lexical variation,morphological/lexical,minor effect; segment remains highly stable,Different verbal form.
9,5,principes,pricipes,graphic/orthographic variation,orthographic,lowers Jaccard similarity,Different spelling of the same lexical item.


In [59]:
df_variant_type_counts = (
    df_variant_classification["variation_type"]
    .value_counts()
    .reset_index()
)

df_variant_type_counts.columns = ["variation_type", "count"]

df_variant_type_counts

,variation_type,count
0,morphological variation,2
1,morphological/graphic variation,2
2,graphic/orthographic variation,2
3,lexical variation,1
4,graphic/morphological variation,1
5,omission/compression,1
6,morphological/lexical variation,1
7,lexical or abbreviation-related variation,1
8,addition,1
9,word order variation,1


In [60]:
df_linguistic_level_counts = (
    df_variant_classification["linguistic_level"]
    .value_counts()
    .reset_index()
)

df_linguistic_level_counts.columns = ["linguistic_level", "count"]

df_linguistic_level_counts

,linguistic_level,count
0,morphological,2
1,morphological/orthographic,2
2,textual,2
3,orthographic,2
4,lexical/textual,1
5,orthographic/morphological,1
6,morphological/lexical,1
7,lexical/orthographic,1
8,syntactic/sequential,1
9,orthographic/lexical,1


In [61]:
df_variant_classification.to_csv("variant_classification.csv", index=False, encoding="utf-8")
files.download("variant_classification.csv")

df_variant_type_counts.to_csv("variant_type_counts.csv", index=False, encoding="utf-8")
files.download("variant_type_counts.csv")

df_linguistic_level_counts.to_csv("linguistic_level_counts.csv", index=False, encoding="utf-8")
files.download("linguistic_level_counts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 9.1 Interpretazione della classificazione

La classificazione delle varianti mostra che la variazione tra i due testimoni non riguarda un solo livello linguistico. Alcune differenze sono grafiche o ortografiche, come *quattuor / quator* e *principes / pricipes*; altre riguardano forme morfologiche, come *instituunt / instituerunt* e *sacrorum / sacrum*; in altri casi abbiamo delle omissioni.

La linguistic analysis, dunque, funziona come ponte tra il dato computazionale e la lettura filologica: organizza le differenze osservate in categorie esplicite e rende più trasparente il modo in cui il workflow produce interpretazione.

## 10. Conclusioni

All'interno di questo progetto abbiamo dimostrato come codifica TEI, normalizzazione, misure di similarità, edit distance, allineamento e classificazione linguistica possano essere combinati in un piccolo workflow riproducibile per lo studio della variazione testuale tra due testimoni manoscritti. In *Nota quod sacro concilio non est detrahendum* di Johannes Hildessen la principale scelta metodologica è stata quella di confrontare i due testimoni a livello di segmentazione argomentativa poichè offre unità testuali più piccole e più facilmente comparabili.

La valutazione della similarità ha mostrato che i sei segmenti non presentano lo stesso grado di stabilità. Gli argomenti 2, 4 e 6 mostrano valori alti di similarità e conservano una forte continuità lessicale e sequenziale tra i due testimoni. L’argomento 4 è particolarmente stabile dal punto di vista della sequenza, mentre l’argomento 6 presenta la più alta sovrapposizione lessicale. Al contrario, gli argomenti 1, 3 e 5 mostrano forme di variazione più problematiche. L’argomento 1 è molto breve, quindi anche un numero limitato di differenze incide fortemente sui punteggi di similarità. L’argomento 3 presenta un’elevata sovrapposizione lessicale, ma una similarità sequenziale più bassa, soprattutto perché D54 conserva una formulazione più estesa rispetto a 4941. L’argomento 5 combina variazione lessicale con una formula finale aggiuntiva in 4941.

Successivamente, l’uso complementare di Jaccard similarity, Longest Common Subsequence e Levenshtein distance ha permesso di osservare aspetti diversi della variazione testuale. La Jaccard similarity misura la sovrapposizione lessicale, la LCS misura la continuità sequenziale, mentre la Levenshtein distance rappresenta la variazione come costo di trasformazione tra sequenze di token.

L’allineamento esplorativo Needleman-Wunsch applicato all’argomento 3 ha reso più interpretabili i risultati quantitativi. Ha mostrato che la minore similarità sequenziale di questo segmento è collegata a materiale presente in D54 e assente in 4941, in particolare il passaggio relativo alla condanna delle eresie. Allo stesso tempo, l’allineamento ha evidenziato anche un limite metodologico: quando compaiono formule ripetute, l’algoritmo può scegliere una corrispondenza possibile tra più corrispondenze plausibili.

La classificazione finale delle varianti ha collegato i risultati computazionali all’interpretazione linguistica e filologica. La variazione tra i due testimoni riguarda più livelli: variazione grafica e ortografica, variazione morfologica, variazione lessicale, aggiunte, omissioni o compressioni, e variazioni nell’ordine delle parole. Questo conferma che i punteggi di similarità numerici devono essere interpretati qualitativamente perchè un punteggio basso non indica sempre lo stesso tipo di variazione, e un punteggio alto può comunque nascondere differenze locali rilevanti dal punto di vista filologico.

Il progetto presenta diversi limiti, in primo luogo, il dataset è volutamente ridotto: include solo due testimoni e sei segmenti argomentativi per rendere il workflow gestibile ma non consente di formulare conclusioni generali sull’intera tradizione manoscritta. In secondo luogo, la classificazione delle varianti è manuale e interpretativa, quindi si basa sulle differenze osservate e sugli output computazionali. Infine, l’allineamento è basato su token e utilizza uno schema di punteggio semplice, quindi non può rappresentare tutta la complessità della variazione nel latino medievale.

Ad ogni modo, il lavoro riesce a dimostrare come gli strumenti computazionali possano supportare l’interpretazione filologica e rappresenta un primo passo verso un’edizione critica dei testimoni di *Nota quod sacro concilio non est detrahendum* di Johannes Hildessen.